# Part -1 (class work) : Text Pre-processing in NLP.

# Basics of Text Data Cleaning.
## Instructions and Requirements:

In this Notebook we will evaluate few basic text data cleaning techniques which are most common for any NLP tasks.

This Notebook makes use of "NLTK" and "Regex" Library a lot.

Dataset: "trump_tweets.csv"


## Step 1: Install & Import Required Libraries

In [8]:
import pandas as pd
import numpy as np
import re
import nltk

# Download required NLTK data
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('punkt')
nltk.download('omw-1.4')

from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer, PorterStemmer

stop_words = set(stopwords.words('english'))
wordnet = WordNetLemmatizer()
porter = PorterStemmer()

print("Libraries loaded successfully!")

Libraries loaded successfully!


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


## Step 2: Helper Functions for Text Cleaning

In [9]:
def lower_order(text):
    """Convert text to lowercase."""
    return text.lower()


def remove_urls(text):
    """Remove URLs (http, https, www links and t.co short links)."""
    text = re.sub(r'http\S+|www\.\S+|https\S+', '', text)
    return text


def remove_emoji(text):
    """Remove emojis and unicode characters."""
    # Broad unicode emoji range
    emoji_pattern = re.compile(
        "["
        u"\U0001F600-\U0001F64F"  # emoticons
        u"\U0001F300-\U0001F5FF"  # symbols & pictographs
        u"\U0001F680-\U0001F6FF"  # transport & map
        u"\U0001F1E0-\U0001F1FF"  # flags
        u"\U00002700-\U000027BF"  # dingbats
        u"\U000024C2-\U0001F251"
        "]+",
        flags=re.UNICODE
    )
    return emoji_pattern.sub('', text)


def removeunwanted_characters(text):
    """Remove mentions (@user), hashtags, punctuation, and special characters."""
    text = re.sub(r'@\w+', '', text)          # Remove mentions
    text = re.sub(r'#\w+', '', text)          # Remove hashtags
    text = re.sub(r'RT\s+', '', text)         # Remove retweet marker
    text = re.sub(r'[^a-z\s]', '', text)      # Keep only letters and spaces
    text = re.sub(r'\s+', ' ', text).strip()  # Collapse multiple spaces
    return text


print("Helper functions defined!")

Helper functions defined!


## Step 3: Text Cleaning Pipeline

The pipeline accepts a **single string** and applies all cleaning steps in order:
1. Lowercase
2. Remove URLs
3. Remove emojis
4. Remove unwanted characters (mentions, hashtags, punctuation)
5. Tokenize
6. Remove stopwords
7. Lemmatize or Stem (based on the `rule` parameter)

In [10]:
def text_cleaning_pipeline(dataset, rule="lemmatize"):
    """
    A full text preprocessing pipeline for a single string.

    Parameters:
    -----------
    dataset : str
        The raw input text (a single tweet or document).
    rule : str
        Either 'lemmatize' (default) or 'stem'.

    Returns:
    --------
    str : cleaned, tokenized, and normalized text joined as a string.
    """
    # Convert the input to lowercase
    data = lower_order(dataset)

    # Remove URLs
    data = remove_urls(data)

    # Remove emojis
    data = remove_emoji(data)

    # Remove all other unwanted characters (mentions, hashtags, punctuation)
    data = removeunwanted_characters(data)

    # Create tokens by splitting on whitespace
    tokens = data.split()

    # Remove stopwords
    tokens = [word for word in tokens if word not in stop_words]

    if rule == "lemmatize":
        # Lemmatize each token (pos='v' treats words as verbs for better results)
        tokens = [wordnet.lemmatize(word, pos='v') for word in tokens]
    elif rule == "stem":
        # Stem each token using Porter Stemmer
        tokens = [porter.stem(word) for word in tokens]
    else:
        print("Pick between lemmatize or stem")

    return " ".join(tokens)

## Step 4: Test the Pipeline on a Sample Tweet

In [11]:
sample = "Hello @gabe_flomo 👋🏾, I still want us to hit that new sushi spot??? LMK when you're free cuz I can't go this or next weekend since I'll be swimming!!! #sushiBros #rawFish #🍱"

print("Original:")
print(sample)
print()
print("After lemmatize pipeline:")
print(text_cleaning_pipeline(sample, rule="lemmatize"))
print()
print("After stem pipeline:")
print(text_cleaning_pipeline(sample, rule="stem"))

Original:
Hello @gabe_flomo 👋🏾, I still want us to hit that new sushi spot??? LMK when you're free cuz I can't go this or next weekend since I'll be swimming!!! #sushiBros #rawFish #🍱

After lemmatize pipeline:
hello still want us hit new sushi spot lmk youre free cuz cant go next weekend since ill swim

After stem pipeline:
hello still want us hit new sushi spot lmk your free cuz cant go next weekend sinc ill swim


## Step 5: Load Dataset and Apply the Pipeline

In [12]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [13]:
# Load the trump tweets dataset
df = pd.read_csv("drive/MyDrive/AI and Machine Learning/workshops/workshop8/trumptweets_small.csv")
print(df.shape)
df.head()

(41122, 9)


,id,link,content,date,retweets,favorites,mentions,hashtags,geo
0,1698308935,https://twitter.com/realDonaldTrump/status/169...,Be sure to tune in and watch Donald Trump on L...,2009-05-04 20:54:25,500,868,NaN,NaN,NaN
1,1701461182,https://twitter.com/realDonaldTrump/status/170...,Donald Trump will be appearing on The View tom...,2009-05-05 03:00:10,33,273,NaN,NaN,NaN
2,1737479987,https://twitter.com/realDonaldTrump/status/173...,Donald Trump reads Top Ten Financial Tips on L...,2009-05-08 15:38:08,12,18,NaN,NaN,NaN
3,1741160716,https://twitter.com/realDonaldTrump/status/174...,New Blog Post: Celebrity Apprentice Finale and...,2009-05-08 22:40:15,11,24,NaN,NaN,NaN
4,1773561338,https://twitter.com/realDonaldTrump/status/177...,"""My persona will never be that of a wallflower...",2009-05-12 16:07:28,1399,1965,NaN,NaN,NaN


In [14]:
# Apply the cleaning pipeline to the 'content' column
# Using lambda so we can pass each row as a string
df['cleaned_content'] = df['content'].dropna().apply(
    lambda text: text_cleaning_pipeline(text, rule="lemmatize")
)

print("Before cleaning:")
print(df['content'][0])
print()
print("After cleaning:")
print(df['cleaned_content'][0])

Before cleaning:
Be sure to tune in and watch Donald Trump on Late Night with David Letterman as he presents the Top Ten List tonight!

After cleaning:
sure tune watch donald trump late night david letterman present top ten list tonight


In [15]:
# View a sample of cleaned vs original text
df[['content', 'cleaned_content']].head(10)

,content,cleaned_content
0,Be sure to tune in and watch Donald Trump on L...,sure tune watch donald trump late night david ...
1,Donald Trump will be appearing on The View tom...,donald trump appear view tomorrow morning disc...
2,Donald Trump reads Top Ten Financial Tips on L...,donald trump read top ten financial tip late s...
3,New Blog Post: Celebrity Apprentice Finale and...,new blog post celebrity apprentice finale less...
4,"""My persona will never be that of a wallflower...",persona never wallflower id rather build wall ...
5,"Miss USA Tara Conner will not be fired - ""I've...",miss usa tara conner fire ive always believer ...
6,Listen to an interview with Donald Trump discu...,listen interview donald trump discuss new book...
7,"""Strive for wholeness and keep your sense of w...",strive wholeness keep sense wonder intact dona...
8,"Enter the ""Think Like A Champion"" signed book ...",enter think like champion sign book keychain c...
9,"""When the achiever achieves, it's not a platea...",achiever achieve plateau begin donald j trump
